# **Regression: Bagging Regressor**

## **Justification of Preprocessing Strategy**

### **The Distinction Between Bagging and Random Forest**
Standard Scikit-Learn `BaggingRegressor` implementations default to using a `DecisionTreeRegressor` as the base estimator. However, utilizing a Decision Tree inside a Bagging algorithm simply recreates a Random Forest, which we have already independently optimized. To ensure architectural diversity, we treat the Bagging Regressor as a **Dependent Ensemble**, injecting one of our previously optimized mathematical champions as the base estimator. We selected our champion **Ridge Regressor (`alpha=0.01`)**.

### **Scale Dependency of the Base Estimator**
Because the "brain" of our Bagging Regressor is a Ridge model—which relies on L2 regularization and geometric coefficients—the entire ensemble inherits its scale sensitivity. If we fed unscaled data into the Bagging Regressor, the internal Ridge models would fail to properly regularize large-magnitude features. Therefore, we must rigorously apply **Standardization** to the numerical features before passing the data to the ensemble. 

## **Experiment Design**

The hyperparameter optimization for this ensemble does not alter the intelligence of the base estimator (Ridge's `alpha` remains locked at 0.01). Instead, we optimize the **logistics of the ensemble**: how many clones to create (`n_estimators`), how much data to feed each clone (`max_samples`), and feature subsetting (`max_features`) to force the Ridge models to think differently. 

We log **both Train and Test metrics (RMSE, MAE, R²)** across 3 levels of optimization:

* **Baseline**: 10 Ridge clones using 100% of the training data (bootstrap enabled).
* **GridSearchCV**: A 3-fold cross-validated search exploring discrete steps in ensemble size and data starvation.
* **Optuna Optimization**: Bayesian search exploring the continuous fraction of `max_samples` and `max_features` to maximize ensemble diversity and minimize the validation RMSE.

In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Bagging")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 
# Drop classification targets to prevent data leakage!
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    # Logs Train and Test metrics explicitly to monitor the Overfitting Gap
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)
    
# Scaling is MANDATORY because the base estimator is Ridge
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# Locking in the Champion Base Estimator
champion_base = Ridge(alpha=0.01, random_state=SEED)

# ---------------------------------------------------------
# RUN 1: BASELINE
# ---------------------------------------------------------
with mlflow.start_run(run_name="Bagging_Reg_Baseline"):
    reg_base = BaggingRegressor(
        estimator=champion_base,
        n_estimators=10, # default
        random_state=SEED,
        n_jobs=-1
    )
    
    start_time = time.time()
    reg_base.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    # Explicit Predictions
    y_pred_train_base = reg_base.predict(X_train_scaled)
    y_pred_test_base = reg_base.predict(X_test_scaled)
    
    mlflow.log_params(reg_base.get_params())
    mlflow.log_param("optimization", "none_default")
    mlflow.log_param("base_estimator", "Ridge_alpha_0.01")
    
    log_regression_metrics(y_train, y_pred_train_base, y_test, y_pred_test_base, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV
# ---------------------------------------------------------
with mlflow.start_run(run_name="Bagging_Reg_GridSearch"):
    param_grid = {
        "n_estimators": [50, 100],
        "max_samples": [0.6, 0.8, 1.0],
        "max_features": [0.8, 1.0]
    }

    grid_reg = GridSearchCV(
        estimator=BaggingRegressor(estimator=champion_base, random_state=SEED, n_jobs=-1),
        param_grid=param_grid,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    start_time = time.time()
    grid_reg.fit(X_train_scaled, y_train)
    duration = time.time() - start_time

    best_bag_grid = grid_reg.best_estimator_
    
    # Explicit Predictions
    y_pred_train_grid = best_bag_grid.predict(X_train_scaled)
    y_pred_test_grid = best_bag_grid.predict(X_test_scaled)

    mlflow.log_params(grid_reg.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    mlflow.log_param("base_estimator", "Ridge_alpha_0.01")
    
    log_regression_metrics(y_train, y_pred_train_grid, y_test, y_pred_test_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective_reg(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
        "max_features": trial.suggest_float("max_features", 0.7, 1.0)
    }

    model = BaggingRegressor(estimator=champion_base, **params, random_state=SEED, n_jobs=-1)
    
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), 
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    return -scores.mean()

with mlflow.start_run(run_name="Bagging_Reg_Optuna"):
    study_reg = optuna.create_study(direction="minimize")
    
    start_time = time.time()
    study_reg.optimize(objective_reg, n_trials=15)
    duration = time.time() - start_time

    best_bag_optuna = BaggingRegressor(estimator=champion_base, **study_reg.best_params, random_state=SEED, n_jobs=-1)
    best_bag_optuna.fit(X_train_scaled, y_train)
    
    # Explicit Predictions
    y_pred_train_optuna = best_bag_optuna.predict(X_train_scaled)
    y_pred_test_optuna = best_bag_optuna.predict(X_test_scaled)

    mlflow.log_params(study_reg.best_params)
    mlflow.log_param("optimization", "optuna")
    mlflow.log_param("base_estimator", "Ridge_alpha_0.01")
    
    log_regression_metrics(y_train, y_pred_train_optuna, y_test, y_pred_test_optuna, duration)

2026/05/22 17:13:57 INFO mlflow.tracking.fluent: Experiment with name 'Regression_Bagging' does not exist. Creating a new experiment.



--- Starting Data Scaling (Standardization) ---


[I 2026-05-22 17:15:30,663] A new study created in memory with name: no-name-70324c55-e3d9-4b92-adb9-ee89fbce831b
[I 2026-05-22 17:15:37,058] Trial 0 finished with value: 1.0499387596433154 and parameters: {'n_estimators': 75, 'max_samples': 0.9190432370883941, 'max_features': 0.8412234818927586}. Best is trial 0 with value: 1.0499387596433154.
[I 2026-05-22 17:15:42,673] Trial 1 finished with value: 1.1722803272558158 and parameters: {'n_estimators': 70, 'max_samples': 0.5612073740060133, 'max_features': 0.8244523377684556}. Best is trial 0 with value: 1.0499387596433154.
[I 2026-05-22 17:15:51,287] Trial 2 finished with value: 1.5714661935492247 and parameters: {'n_estimators': 124, 'max_samples': 0.7088941480972663, 'max_features': 0.7453605064362568}. Best is trial 0 with value: 1.0499387596433154.
[I 2026-05-22 17:15:56,478] Trial 3 finished with value: 1.5311480600551235 and parameters: {'n_estimators': 69, 'max_samples': 0.542808898289872, 'max_features': 0.7697700189362714}. Be

## Winner Run Selection 

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| Bagging_Reg_Baseline | 0.39597 | 0.40321 | 0.69369 | 0.71137 | 0.99413 | 0.99387 | 5.62s |
| Bagging_Reg_GridSearch | 0.39638 | 0.40357 | 0.69367 | 0.71127 | 0.99413 | 0.99387 | 84.88s |
| Bagging_Reg_Optuna | 0.40283 | 0.41014 | 0.69418 | 0.71175 | 0.99412 | 0.99386 | 173.41s |

### Generalization Check (Test − Train)
- **Bagging_Reg_Baseline:** MAE gap = 0.40321 − 0.39597 = **+0.00723** and RMSE gap = 0.71137 − 0.69369 = **+0.01768** → PASS.
- **Bagging_Reg_GridSearch:** MAE gap = 0.40357 − 0.39638 = **+0.00719** and RMSE gap = 0.71127 − 0.69367 = **+0.01760** → PASS.
- **Bagging_Reg_Optuna:** MAE gap = 0.41014 − 0.40283 = **+0.00731** and RMSE gap = 0.71175 − 0.69418 = **+0.01757** → PASS.

### Overfitting / Underfitting Validation
- None of the runs shows overfitting. The Train/Test gaps are very small and stable.
- None of the runs shows underfitting. All Test R² values are very high, so the ensemble is learning the target structure properly.
- There is no evidence of catastrophic RMSE growth relative to MAE.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: all three runs.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- Bagging_Reg_Baseline: 0.40321
- Bagging_Reg_GridSearch: 0.40357
- Bagging_Reg_Optuna: 0.41014
- Lowest MAE: **Bagging_Reg_Baseline**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- Bagging_Reg_GridSearch: 0.71127
- Bagging_Reg_Baseline: 0.71137
- Bagging_Reg_Optuna: 0.71175
- The GridSearch run is marginally better on RMSE, but the difference is tiny.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- Bagging_Reg_Baseline: 0.99387
- Bagging_Reg_GridSearch: 0.99387
- Bagging_Reg_Optuna: 0.99386
- The values are effectively tied.

### Final Decision
**Winner: Bagging_Reg_Baseline**

**Justification:** `Bagging_Reg_Baseline` is the best overall choice among the runs that pass the generalization filter because it has the lowest Test MAE and an effectively tied Test R², while fit time is dramatically lower than the optimized runs. The GridSearch run is only marginally better in RMSE, but not enough to overturn the MAE priority.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **base_estimator** | Ridge_alpha_0.01 |
| **n_estimators** | 10 |
| **max_samples** | 1.0 |
| **max_features** | 1.0 |
| **random_state** | 42 |
| **n_jobs** | -1 |

## Overfitting / Underfitting Diagnosis
- All three runs generalize well and none is disqualified.
- The baseline is slightly better on the primary metric and is also the most efficient operationally.
- The optimized runs do not provide enough improvement to justify their higher compute cost.